<a href="https://colab.research.google.com/github/rhyan10/X-MACE/blob/X-MACE_socs/tutorials/tutorial-summer-school.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ASE Tutorial: Opening and Inspecting the XYZ File

This notebook walks through:

1. Setting up the environment and getting the data
2. Reading an extended XYZ file with ASE
3. Exploring basic `Atoms` properties
4. Understanding the `atoms.info` dictionary
5. Checking the correct shape of every data array
6. Iterating over all frames

The file is in **extended XYZ (extXYZ)** format — each frame's comment line carries
the reference properties, which ASE parses automatically into `atoms.info`.

## 0. Setup

Colab starts from a clean machine every time, so we install ASE and fetch the dataset
here. Run this cell first.

In [ ]:
# ASE is not preinstalled in Colab
!pip install -q ase

import ase, ase.io
print("ASE version:", ase.__version__)

### 0.1 Get the dataset

Three ways to get `SINGLET_SOC_ALL.xyz` onto the machine. The cell below tries them
in order, so it works whether or not you have a direct download link.

* **Best for a tutorial:** host the file (or a trimmed version) somewhere with a direct
  link and fill in `DATA_URL` below — then nobody has to do anything manually.
* **Fallback:** the cell will prompt you to upload the file from your laptop.
* The full datasets are also on
  [figshare](https://figshare.com/articles/dataset/Datasets_for_X-MACE_Publication/28425173).

In [ ]:
import os

DATA_FILE = "SINGLET_SOC_ALL.xyz"
DATA_URL  = ""   # <-- paste a direct download link here (raw GitHub, release asset, etc.)

if os.path.exists(DATA_FILE):
    print(f"Found {DATA_FILE} already.")
elif DATA_URL:
    !wget -q -O {DATA_FILE} {DATA_URL}
    print(f"Downloaded {DATA_FILE}")
else:
    print("No URL set — please upload the file from your computer.")
    from google.colab import files
    files.upload()

assert os.path.exists(DATA_FILE), f"{DATA_FILE} is still missing — cannot continue."
print("Size on disk:", round(os.path.getsize(DATA_FILE) / 1e6, 2), "MB")

## 1. Imports

In [ ]:
import ase.io
import numpy as np

## 2. Reading the XYZ File

Pass `":"` as the index to read **all** frames, or `":10"` for just the first 10
(much faster while you are exploring a large file).

In [ ]:
# Read all frames — returns a list of Atoms objects
db = ase.io.read(DATA_FILE, ":")

print(f"Total frames loaded : {len(db)}")

# Work with the first frame for exploration
atoms = db[0]
print(f"Type of each element: {type(atoms)}")

## 3. Basic Atoms Properties

In [ ]:
print("Number of atoms    :", len(atoms))
print("Chemical symbols   :", list(atoms.symbols))
print("Positions shape    :", atoms.positions.shape)   # (N_atoms, 3)

## 4. Listing All `atoms.info` Keys

`atoms.info` is a plain Python `dict`. Every key in the extXYZ comment line ends up
here, already parsed into arrays.

In [ ]:
print("Keys in atoms.info:\n")
for k, v in atoms.info.items():
    arr = np.asarray(v)
    print(f"  {k:22s}  shape={str(arr.shape):16s}  dtype={arr.dtype}")

## 5. Shape Reference

This dataset has **4 electronic states** and **N = 23 atoms**.
The number of unique state pairs is `4 x (4-1) / 2 = 6`.

| Key | Expected shape | Meaning |
|---|---|---|
| `REF_energy` | `(1, 4)` | 1 geometry x 4 state energies |
| `REF_forces` | `(23, 4, 3)` | atoms x states x xyz |
| `REF_couplings` | `(23, 6, 3)` | atoms x state pairs x xyz |
| `REF_smooth_nacs` | `(23, 6, 3)` | smoothed NACs, same layout |
| `REF_socs` | `(1, n_soc)` | flat SOC vector |

The shapes are derived from the data in the cells below rather than hard-coded, so
this notebook still works if you swap in a dataset with a different number of atoms
or states.

## 6. Inspecting Each Key

### 6.1 Energy — shape `(1, n_states)`

In [ ]:
energy = np.array(atoms.info['REF_energy'])
print("energy shape :", energy.shape)   # (1, 4)
print("values       :", energy)

n_states = energy.shape[-1]
n_pairs  = n_states * (n_states - 1) // 2
N        = len(atoms)
print(f"\nDerived: N_atoms={N}, n_states={n_states}, n_pairs={n_pairs}")

### 6.2 Forces — shape `(N_atoms, n_states, 3)`

In [ ]:
forces = np.array(atoms.info['REF_forces'])
print("forces shape :", forces.shape)   # (23, 4, 3)

# Re-order to (n_states, N_atoms, 3) if you prefer state-first indexing
forces_sf = forces.transpose(1, 0, 2)
print("state-first  :", forces_sf.shape)

### 6.3 Couplings — shape `(N_atoms, n_pairs, 3)`

Switch the key between `REF_couplings` (raw NACs) and `REF_smooth_nacs`
(the smoothed version X-MACE trains on) to compare them.

In [ ]:
NAC_KEY = 'REF_smooth_nacs'   # or 'REF_couplings'

couplings = np.array(atoms.info[NAC_KEY])
print(f"{NAC_KEY} shape :", couplings.shape)   # (23, 6, 3)

### 6.4 Spin-Orbit Couplings — flat vector

The SOC vector length depends on how many singlet/triplet states the dataset covers,
so read it off the data rather than assuming a number. Whatever prints here is what
you pass to `--soc_num` when training.

In [ ]:
socs = np.array(atoms.info['REF_socs'])
print("socs shape   :", socs.shape)
print("n_soc        :", socs.size, " <- use this value for --soc_num")

## 7. Sanity-Check Script

Verify the shapes for every frame, not just the first one. Missing keys are reported
rather than raising, so this still runs on datasets without SOCs or NACs.

In [ ]:
def expected_shapes(atoms):
    N        = len(atoms)
    n_states = np.array(atoms.info['REF_energy']).shape[-1]
    n_pairs  = n_states * (n_states - 1) // 2
    return {
        'REF_energy'      : (1, n_states),
        'REF_forces'      : (N, n_states, 3),
        'REF_couplings'   : (N, n_pairs,  3),
        'REF_smooth_nacs' : (N, n_pairs,  3),
    }

def check_frame(atoms, verbose=False):
    problems = []
    for key, exp in expected_shapes(atoms).items():
        if key not in atoms.info:
            problems.append(f"{key}: missing")
            continue
        got = np.array(atoms.info[key]).shape
        if got != exp:
            problems.append(f"{key}: expected {exp}, got {got}")
        if verbose:
            mark = "OK " if got == exp else "BAD"
            print(f"  [{mark}] {key:22s} expected {str(exp):15s} got {got}")
    return problems


print("Frame 0 in detail:")
check_frame(db[0], verbose=True)

print("\nScanning all frames...")
bad = {i: p for i, a in enumerate(db) if (p := check_frame(a))}

if not bad:
    print(f"All {len(db)} frames have correct shapes.")
else:
    print(f"{len(bad)} of {len(db)} frames have problems. First few:")
    for i, probs in list(bad.items())[:5]:
        print(f"  frame {i}: {'; '.join(probs)}")

## 8. Iterating Over All Frames

Printing one line per frame gets unwieldy for a large dataset, so this shows a preview
and then summary statistics over everything.

In [ ]:
PREVIEW = 10   # how many frames to print individually

print(f"{'Frame':>6}  {'Min energy':>18}  {'Max |force|':>20}")
print("-" * 50)

min_energies, max_forces = [], []
for i, frame in enumerate(db):
    e = np.array(frame.info['REF_energy'])
    f = np.array(frame.info['REF_forces'])
    min_energies.append(e.min())
    max_forces.append(np.abs(f).max())
    if i < PREVIEW:
        print(f"{i:>6}  {e.min():>18.6f}  {np.abs(f).max():>20.6f}")

if len(db) > PREVIEW:
    print(f"... and {len(db) - PREVIEW} more frames")

min_energies = np.array(min_energies)
max_forces   = np.array(max_forces)

print(f"\nOver all {len(db)} frames:")
print(f"  ground-state energy : {min_energies.min():.6f} to {min_energies.max():.6f}")
print(f"  largest |force|     : {max_forces.max():.6f}  (frame {max_forces.argmax()})")

## 9. Quick Visual Check

A histogram of the energy gap between the two lowest states — near-zero values are the
conical-intersection region that X-MACE is built to handle.

In [ ]:
import matplotlib.pyplot as plt

gaps = np.array([np.sort(np.array(f.info['REF_energy']).ravel())[1]
                 - np.sort(np.array(f.info['REF_energy']).ravel())[0]
                 for f in db])

plt.figure(figsize=(6, 4))
plt.hist(gaps, bins=50)
plt.xlabel("S1 - S0 energy gap")
plt.ylabel("Number of frames")
plt.title("Energy gap distribution")
plt.tight_layout()
plt.show()

print(f"Smallest gap: {gaps.min():.6f} (frame {gaps.argmin()})")